In [1]:
from langchain_openai import ChatOpenAI
import os
from pydantic import SecretStr

llm = ChatOpenAI(
    model="gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=SecretStr(os.environ["OPENROUTER_API_KEY"])
)

# Methods to get Structured Output

## Method 1: Using prompting

In [3]:
from langchain_core.prompts import PromptTemplate

result=llm.invoke("Tell me a joke about programming.Generate the output in key-value pair format with the following keys: setup, punchline")
result.content

'{\n  "setup": "Why do programmers prefer dark mode?",\n  "punchline": "Because light attracts bugs!"\n}'

## Method 2: Using Pydantic Models

In [4]:
from pydantic import BaseModel

class llm_schema(BaseModel):
    setup: str
    punchline: str

In [ ]:
obj = llm_schema(**{"setup":"Why do programmers prefer dark mode?", "punchline":"Because light attracts bugs!"})
print(obj.setup)
print(obj.punchline)

#Pydantic models dont allow any other keys than the ones defined in the model. If you try to add any other key, it will throw an error.

Why do programmers prefer dark mode?
Because light attracts bugs!


In [7]:
from pydantic import BaseModel, Field

class llm_schema(BaseModel):
    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline of the joke")

In [9]:
llm_structured = llm.with_structured_output(llm_schema)
result=llm_structured.invoke("Tell me a joke about programming")
print(result)
print(type(result))

setup='Why do programmers prefer dark mode?' punchline='Because light attracts bugs!'
<class '__main__.llm_schema'>


## Method 3: Using TypedDict

In [10]:
from typing import TypedDict

class llm_schema(TypedDict):
    setup: str
    punchline: str

In [ ]:
obj = llm_schema({"setup":"Why do programmers prefer dark mode?", "punchline":"Because light attracts bugs!"})
obj["setup"]

#TypedDict model on the other hand allows you to add any other keys than the ones defined in the model. If you try to add any other key, it will not throw an error instead it will just ignore the other keys and only keep the keys defined in the model.

'Why do programmers prefer dark mode?'

In [13]:
llm_structured = llm.with_structured_output(llm_schema)

result=llm_structured.invoke("Tell me a joke about programming")
print(result)

{'setup': 'Why do programmers prefer dark mode?', 'punchline': 'Because light attracts bugs!'}
